# 01 – Exploratory Data Analysis (M2)
CFPB narratives archive, Apr–Jul 2024. Loads the cleaned data and splits through the
`cfpb_triage` package (run `make data splits` first) and reproduces the M2 figures.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

from cfpb_triage.analysis.imbalance import imbalance_report
from cfpb_triage.data.load import load_config, resolve
from cfpb_triage.data.split import load_splits, proportion_table, read_cleaned
from cfpb_triage.evaluation.plots import (plot_class_distribution, plot_length_box,
                                          plot_length_hist)
from cfpb_triage.utils.seed import set_seed

cfg = load_config()
set_seed(cfg["seed"])
FIG = resolve("results/figures"); FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
df = read_cleaned()
train, val, test = load_splits()
print(f"cleaned: {len(df):,} distinct complaints | train {len(train):,} | val {len(val):,} | test {len(test):,}")
df.head(3)[["complaint_id", "product", "group_size", "norm_text"]]

## Cleaning funnel and duplicates
Most raw complaints have no narrative (opt-in). About half of the narratives are copies of a
template letter; each duplicate group is kept as one row, with `group_size` = how many copies it had.

In [ ]:
log = json.loads(resolve("results/logs/cleaning_log.json").read_text(encoding="utf-8"))
display(pd.DataFrame(log["steps"])[["step", "rows", "dropped"]])
print("dedupe:", {k: log["dedupe"][k] for k in ("exact_duplicate_rows", "n_multi_row_groups", "largest_group")})
print("collapse:", log["collapse"])
display(df["group_size"].describe().round(1).to_frame("group_size"))

## Class imbalance
Interpretation: <FILL AFTER RUN: majority class and share, imbalance ratio, normalised entropy>.
A stratified-random guesser's Macro F1 is ≈ 1/K, while its accuracy is ≈ Σp², which is why Macro F1 is our primary metric.

In [ ]:
summary, class_tab = imbalance_report(df["product"])
display(pd.Series(summary, name="value").to_frame())
display(class_tab)

In [ ]:
plot_class_distribution(df["product"], FIG / "class_distribution_log.png")
display(Image(filename=str(FIG / "class_distribution_log.png")))

## Narrative length
Interpretation: <FILL AFTER RUN: median tokens; share under 20 tokens; which classes are longer/shorter>.

In [ ]:
df["n_tokens"] = df["norm_text"].str.split().str.len()
display(df["n_tokens"].describe().round(1).to_frame("tokens"))
plot_length_hist(df["n_tokens"], FIG / "narrative_length_hist.png")
plot_length_box(df, FIG / "narrative_length_by_class.png")
display(Image(filename=str(FIG / "narrative_length_hist.png")))
display(Image(filename=str(FIG / "narrative_length_by_class.png")))

## Split check
Class proportions per split should match the full data, and no group may appear in two splits.
Interpretation: <FILL AFTER RUN: largest deviation in pp; groups shared = 0>.

In [ ]:
display(proportion_table(df, train, val, test).round(4))
gid = [set(p["group_id"]) for p in (train, val, test)]
print("groups shared across splits:", len(gid[0] & gid[1]) + len(gid[0] & gid[2]) + len(gid[1] & gid[2]))

sample_path = resolve("results/tables/sample_io.csv")
pd.read_csv(sample_path) if sample_path.exists() else print("run python scripts/03_eda.py first")

In [ ]:
## Takeaways for M3
<FILL AFTER RUN: 3 bullets – imbalance severity, duplicates collapsed to distinct complaints, short-narrative share>